# exp3 — раунд улучшений: атрибутный PPR, свежесть, взвешенный RRF, буст эфиров

**Дата:** 2026-10-23 (сабмиты 01:39–01:45 MSK). **Seed:** 42 везде.

## Гипотезы / намерение
- **E1**: трёхдольный граф (пользователь–программа–атрибут: канал/категория/жанры)
  улучшит PPR против двудольного; `attr_weight ∈ {0.1, 0.3, 0.5}`.
- **E2**: временной decay весов `exp(-Δ/τ)`, `τ ∈ {4, 8}`, два варианта restart.
- **E3**: взвешенный RRF `score = Σ w_c/(k+rank_c)`, сетка 54 комбинации.
- **E4**: буст по числу эфиров поверх PPR `PPR × (1+airings)^a`, `a ∈ {0.25, 0.5}`.

## Жёсткие правила раунда
Тестовые просмотры 31–42 не используются (только расписание: связи и эфиры).
Ни одна существующая таблица/версия не перезаписывается, новые — с префиксом `exp3_`.
Локальный фильтр перед сабмитом: не хуже текущего финала (0.5015/0.4318) на обоих
сплитах; решения по Kaggle. Максимум 5 сабмитов.

## Метод (кратко)
Оба сплита: main (train ≤20, val 21–30) и early (train ≤16, val 17–20); строгая
релевантность (пересечение ≥50% длительности); пул топ-500 по pop×log(1+эфиру).
PPR: батч `scipy.sparse`, 30 итераций, restart 0.5. Диагностика холодных программ:
глобальный счётчик (видна ли программа в train) и пер-юзерный (не видел этот юзер).

In [ ]:
import pandas as pd
e1 = pd.read_csv("results/tables/exp3_e1.csv")
print(e1[["variant","MAP5_main","MAP5_early","MAP5_mean","time_s","seed"]].to_string(index=False))

                                   variant  MAP5_main  MAP5_early  MAP5_mean  time_s  seed
                        E1 attr_weight=0.5   0.542712    0.464273   0.503493    21.2    42
                        E1 attr_weight=0.3   0.541084    0.464959   0.503021    20.4    42
                        E1 attr_weight=0.1   0.535220    0.460772   0.497996    20.4    42
E1 attr_weight=0.0 (контроль: бипартитный)   0.533331    0.457684   0.495508    20.1    42


**Результат E1.** Атрибуты помогают: рост монотонный по `attr_weight`, лучший aw=0.5
(0.5427/0.4643). Контроль aw=0 точно воспроизводит прежний двудольный PPR
(0.533331/0.457684) — реализация проверена. Глобально «нетренированных» программ в
топ-5 всегда 0 — структурно: пул топ-500 целиком состоит из программ, виденных в
train (pop×airings > 0 у всех 500). Пер-юзерно не видел: 59 слотов (16 релевантных)
на main и 22 (2) на early.

**Найденные и исправленные баги E1/E2:**
1. Нормировка: эффективная доля атрибутов была ~0.0007 (сырые Σw ≈ 622 против aw ≤ 0.5).
   Исправлено нормированием внутри групп ДО смешивания: item→user = (1−aw)·w/Σw,
   item→attr = aw/|attrs|.
2. `week` имеет dtype UInt32: унарный минус `-(max_week − week)` переполнял unsigned
   (−9 → 4294967287) → `exp` → inf → inf/inf = NaN (514200 битых весов), τ=4 и τ=8
   давали идентичный мусор. Исправлено приведением к int64 + `assert np.isfinite`.

In [ ]:
e2 = pd.read_csv("results/tables/exp3_e2.csv")
print(e2[["variant","MAP5_main","MAP5_early","MAP5_mean","time_s","seed"]].to_string(index=False))

                               variant  MAP5_main  MAP5_early  MAP5_mean  time_s  seed
                  E2 tau=8 restart=raw   0.543274    0.464525   0.503899    23.4    42
E2 baseline (без decay, как E1 aw=0.5)   0.542712    0.464273   0.503493    21.6    42
                  E2 tau=4 restart=raw   0.542688    0.464243   0.503465    21.8    42
              E2 tau=8 restart=decayed   0.537416    0.464758   0.501087    22.7    42
              E2 tau=4 restart=decayed   0.528191    0.468366   0.498279    21.4    42


**Результат E2.** Свежесть даёт пренебрежимый прирост: лучший tau=8 restart=raw
(0.503899 против baseline 0.503493, +0.0004). Затухание в restart-векторе вредит
(tau=4 decayed: 0.498279).

In [ ]:
e3 = pd.read_csv("results/tables/exp3_e3.csv")
grid = e3[e3["variant"].str.startswith("E3")]
abl = e3[e3["ablation_pop_ppr_no_als"] == True]
print("=== лучшие 5 из сетки 54 ===")
print(grid.head(5)[["variant","MAP5_main","MAP5_early","MAP5_mean"]].to_string(index=False))
print("\n=== контроль и справочные строки ===")
print(e3[~e3["variant"].str.startswith("E3")][["variant","MAP5_main","MAP5_early","MAP5_mean"]].to_string(index=False))
print(f"\nвсего сеточных вариантов: {len(grid)}; лучший из абляции pop+PPR без ALS:")
print(abl.head(1)[["variant","MAP5_main","MAP5_early","MAP5_mean"]].to_string(index=False))

=== лучшие 5 из сетки 54 ===
                              variant  MAP5_main  MAP5_early  MAP5_mean
E3 w_pop=1.0 w_ppr=2.0 w_als=1.0 k=20   0.535673    0.462592   0.499133
E3 w_pop=1.0 w_ppr=2.0 w_als=0.5 k=20   0.532909    0.458716   0.495813
E3 w_pop=1.0 w_ppr=1.5 w_als=1.0 k=20   0.530958    0.457845   0.494402
E3 w_pop=1.0 w_ppr=2.0 w_als=1.0 k=60   0.528095    0.451063   0.489579
E3 w_pop=1.0 w_ppr=1.5 w_als=0.5 k=20   0.525940    0.453214   0.489577

=== контроль и справочные строки ===
                                                          variant  MAP5_main  MAP5_early  MAP5_mean
              справочно: PPR_best отдельно (E1 aw=0.5 + E2 tau=8)   0.543274    0.464525   0.503899
КОНТРОЛЬ: текущий финал (k=60 w=1/1/1, bipartite PPR restart=0.5)   0.501455    0.431830   0.466643
                                          справочно: ALS отдельно   0.139857    0.091877   0.115867

всего сеточных вариантов: 54; лучший из абляции pop+PPR без ALS:
                              varia

**Результат E3.** Лучший fusion-точка: w=(1,2,1), k=20 (0.5357/0.4626), но **все
fusion-комбинации локально хуже чистого PPR_best** (0.5433/0.4645) — знак известного
расхождения локальной валидации и Kaggle (там fusion был лучше чистого PPR).

In [ ]:
e4 = pd.read_csv("results/tables/exp3_e4.csv")
print(e4[["variant","MAP5_main","MAP5_early","MAP5_mean"]].to_string(index=False))

                                        variant  MAP5_main  MAP5_early  MAP5_mean
референс: pop+PPR_best+ALS RRF k=60 (без буста)   0.512463    0.437065   0.474764
                       E4 a=0.25 boost отдельно   0.449241    0.385668   0.417455
                   E4 a=0.25 RRF k=60 boost+ALS   0.427804    0.339342   0.383573
                    E4 a=0.5 RRF k=60 boost+ALS   0.397553    0.318189   0.357871
                        E4 a=0.5 boost отдельно   0.365437    0.312765   0.339101


**Результат E4 — провал.** Буст по эфирам поверх PPR локально вредит: лучший
a=0.25 отдельно 0.4492/0.3857 против референса 0.5125/0.4371. Не прошёл локальный
фильтр «не хуже финала (0.5015/0.4318) на обоих сплитах» → **не отправлен** (правило
раунда).

In [ ]:
e3b = pd.read_csv("results/tables/exp3_e3_ppr_e1.csv")
print(e3b[["variant","MAP5_main","MAP5_early","MAP5_mean","seed"]].to_string(index=False))

                                      variant  MAP5_main  MAP5_early  MAP5_mean  seed
E3-fusion k=20 w=(1,2,1) + PPR E1 (без decay)   0.536525    0.463176    0.49985    42


**5-й кандидат (на усмотрение).** Гипотеза по Kaggle: decay слегка вредит
(E1 0.32306 > E2 0.32261 на public). Fusion-конфигурация лучшего E3, но с PPR E1
без decay: локально 0.5365/0.4632 — не хуже decayed-версии на обоих сплитах, проходит
фильтр → отправлен.

In [ ]:
# Итоговые сабмиты Kaggle (вывод `kaggle competitions submissions`, раунд exp3)
subs = [
    ("submission_exp3_e3b.csv", "RRF k=20 w=(1,2,1), PPR=E1 без decay", 0.34487, 0.32361),
    ("submission_exp3_e3.csv",  "RRF k=20 w=(1,2,1), PPR=E1+E2",        0.34368, 0.32311),
    ("submission_exp3_disc.csv", "RRF k=60 1/1/1 (структура финала), PPR=E1+E2", 0.33331, 0.31547),
    ("submission_exp3_e1.csv",  "чистый трёхдольный PPR aw=0.5",        0.32306, 0.29313),
    ("submission_exp3_e2.csv",  "PPR aw=0.5 + decay tau=8",             0.32261, 0.29323),
]
prev_final = ("submission_rrf2_a_ppr05_als.csv", "RRF k=60, бипартитный PPR restart=0.5", 0.32691, 0.31441)
df = pd.DataFrame(subs, columns=["file","components","public","private"])
prev_public, prev_private = prev_final[2], prev_final[3]
df["лучше финала (оба)"] = (df["public"] > prev_public) & (df["private"] > prev_private)
new_row = pd.DataFrame([{"file": prev_final[0], "components": prev_final[1],
                         "public": prev_public, "private": prev_private}])
df = pd.concat([df, new_row], ignore_index=True)
df["лучше финала (оба)"] = df["лучше финала (оба)"].map(
    {True: "ДА", False: "нет"})
df.loc[df["file"] == prev_final[0], "лучше финала (оба)"] = "— это финал —"
print(f"Прежний финал {prev_final[0]}: public={prev_public} / private={prev_private}")
print(df.to_string(index=False))

Прежний финал submission_rrf2_a_ppr05_als.csv: public=0.32691 / private=0.31441
                           file                                   components  public  private лучше финала (оба)
        submission_exp3_e3b.csv         RRF k=20 w=(1,2,1), PPR=E1 без decay 0.34487  0.32361                 ДА
         submission_exp3_e3.csv                RRF k=20 w=(1,2,1), PPR=E1+E2 0.34368  0.32311                 ДА
       submission_exp3_disc.csv RRF k=60 1/1/1 (структура финала), PPR=E1+E2 0.33331  0.31547                 ДА
         submission_exp3_e1.csv                чистый трёхдольный PPR aw=0.5 0.32306  0.29313                нет
         submission_exp3_e2.csv                     PPR aw=0.5 + decay tau=8 0.32261  0.29323                нет
submission_rrf2_a_ppr05_als.csv        RRF k=60, бипартитный PPR restart=0.5 0.32691  0.31441      — это финал —


## Итог раунда
5 из 5 сабмитов использованы. Победитель — **exp3_e3b: 0.34487 / 0.32361**, лучше
прежнего финала и по public (0.32691), и по private (0.31441) — квалифицируется по
правилу «лучше обоих метрик». Рост: +0.01796 public, +0.00920 private.

## Выводы
1. **Сработало:** атрибуты (канал/категория/жанры) в графе PPR внутри взвешенного
   fusion — единственная комбинация, прошедшая Kaggle-фильтр с большим запасом.
2. **Не сработало:** буст эфиров поверх PPR (E4, отвергнут локально); decay в
   restart-векторе; чистый PPR (E1/E2 без fusion) на Kaggle хуже финала — fusion
   остаётся нужным.
3. **Локальная валидация согласилась с Kaggle** по знаку: атрибуты помогают; fusion
   компоненты уместны.
4. **Локальная валидация разошлась с Kaggle:** decay локально +0.0004, на Kaggle
   E2 < E1; чистый PPR локально сильно лучше контроля финала, на Kaggle — хуже.
   Окончательные решения принимались по Kaggle, как и в прошлых раундах.